# 3D reporter timelapse — 03_ilastik_mask_postprocessing_and_qc

**Feeds:** Fig 5h

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# Ilastik Mask Post-Processing And QC

This notebook reviews the first full-dataset organoid masks derived from the ilastik pixel probabilities.

Goals:

- verify that the fixed post-processing rule behaves sensibly across the dataset
- separate `soft review flags` from `analytic exclusions` like organoids growing out of frame
- catch local undersegmentation, where a thin cyst tip gets clipped even though the main body stays tracked
- show the worst remaining non-artifact, non-border-hit examples directly in the notebook
- make an image-first case for whether the masks are accurate enough to proceed


## Interpretation Notes

This notebook treats three categories differently:

- `good masks`: the classifier and post-processing are both behaving well
- `soft review flags`: frames automatically flagged for a closer look, not frames that already failed QC
- `off-frame analytic issues`: masks that may be technically accurate, but the organoid grows out of frame and may become unsuitable for aggregate quantification

Important:

- the notebook excludes obvious image-junk artifact windows from the accuracy argument
- the notebook also excludes persistent border-hit windows from the accuracy argument, because those are downstream analysis-scope issues rather than mask-learning failures
- the notebook now checks local frame-to-frame boundary shifts, so small clipped-tip errors can be surfaced even if the total mask area barely changes


## Setup

This section loads the analysis environment, project paths, QC tables, and helper functions used throughout the notebook.


### Load Analysis Packages


In [ ]:
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile as tiff
from IPython.display import Markdown, display
from skimage import morphology
from skimage.registration import phase_cross_correlation

pd.set_option("display.max_columns", 200)
plt.rcParams["figure.dpi"] = 120


### Load Project Paths And QC Inputs


In [ ]:
# -------------------------------
# Project configuration
# -------------------------------
cwd = Path.cwd().resolve()
root_candidates = [cwd] + list(cwd.parents[:3])
ROOT = None
for candidate in root_candidates:
    if (candidate / "data").exists() and (candidate / "results").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Could not locate project root from current working directory.")

DATASET_DIR = ROOT / "data/raw/20260128_BMP4-reporter_LPM-organoids/d2-d5"
POSITION_MANIFEST_PATH = ROOT / "results/manifests/acquisition_position_manifest.tsv"
PROBABILITY_ROOT = ROOT / "results/ilastik/pixel_probabilities/full_dataset_v1"
MASK_ROOT = ROOT / "results/ilastik/organoid_masks/full_dataset_v1"
METRICS_PATH = ROOT / "results/ilastik/qc/full_dataset_v1_mask_metrics.tsv"
POSITION_SUMMARY_PATH = ROOT / "results/ilastik/qc/full_dataset_v1_mask_position_summary.tsv"
PREVIEW_DIR = ROOT / "results/previews/03_ilastik_mask_postprocessing_qc"
TABLE_DIR = ROOT / "results/tables"
FIGURE_DIR = ROOT / "results/figures/03"
STACK_DIR = ROOT / "results/ilastik/qc/phase_mask_overlay_stacks/full_dataset_v1"

PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
STACK_DIR.mkdir(parents=True, exist_ok=True)

position_manifest = pd.read_csv(POSITION_MANIFEST_PATH, sep="\t")
metrics_df = pd.read_csv(METRICS_PATH, sep="\t")
position_summary = pd.read_csv(POSITION_SUMMARY_PATH, sep="\t")
INTERVAL_HOURS = float(position_manifest["interval_ms"].dropna().iloc[0]) / 3_600_000.0
TIME_DISPLAY_OFFSET_HOURS = 48.0

POSITION_RE = re.compile(r"Pos(?P<position_index>\d+)$")

print("Project root:", ROOT)
print("Metrics rows:", len(metrics_df))
print("Preview dir:", PREVIEW_DIR)
print("Stack dir:", STACK_DIR)


### Define Helper Functions


In [ ]:
# -------------------------------
# Helpers
# -------------------------------
def position_index_from_label(position_label: str) -> int:
    match = POSITION_RE.fullmatch(position_label)
    if not match:
        raise ValueError(f"Unexpected position label: {position_label}")
    return int(match.group("position_index"))


def display_time_hours_from_index(time_index: int | float) -> float:
    return float(time_index) * INTERVAL_HOURS + TIME_DISPLAY_OFFSET_HOURS


def format_display_hours_from_index(time_index: int | float, decimals: int = 1) -> str:
    return f"{display_time_hours_from_index(time_index):.{decimals}f} h"


def display_time_df(df: pd.DataFrame) -> pd.DataFrame:
    output = df.copy()
    rename_map = {
        "time_index": "time_hours",
        "previous_time_index": "previous_time_hours",
        "persistent_border_touch_onset": "persistent_border_touch_onset_hours",
    }
    for column, display_column in rename_map.items():
        if column in output.columns:
            output[display_column] = output[column].map(display_time_hours_from_index)
    return output


def phase_path(position_label: str, time_index: int) -> Path:
    position_index = position_index_from_label(position_label)
    return (
        DATASET_DIR
        / position_label
        / f"img_channel000_position{position_index:03d}_time{time_index:09d}_z000.tif"
    )


def probability_path(position_label: str, time_index: int) -> Path:
    position_index = position_index_from_label(position_label)
    return (
        PROBABILITY_ROOT
        / position_label
        / f"img_channel000_position{position_index:03d}_time{time_index:09d}_z000_Probabilities.tiff"
    )


def mask_path(position_label: str, time_index: int) -> Path:
    position_index = position_index_from_label(position_label)
    return (
        MASK_ROOT
        / position_label
        / f"img_channel000_position{position_index:03d}_time{time_index:09d}_z000_mask.tiff"
    )


def load_phase(position_label: str, time_index: int) -> np.ndarray:
    return tiff.imread(phase_path(position_label, time_index))


def load_probability(position_label: str, time_index: int) -> np.ndarray:
    return tiff.imread(probability_path(position_label, time_index))[..., 0]


def load_mask(position_label: str, time_index: int) -> np.ndarray:
    return tiff.imread(mask_path(position_label, time_index)).astype(bool)


def display_image(image: np.ndarray, low_q: float = 1.0, high_q: float = 99.0) -> np.ndarray:
    low, high = np.percentile(image, [low_q, high_q])
    if math.isclose(high, low):
        high = low + 1.0
    return np.clip((image - low) / (high - low), 0, 1)


def draw_mask(ax, mask: np.ndarray, color: str = "deepskyblue", linewidth: float = 1.8) -> None:
    if mask.any():
        ax.contour(mask.astype(float), levels=[0.5], colors=[color], linewidths=linewidth)


def estimate_global_phase_shift(prev_image: np.ndarray, current_image: np.ndarray) -> tuple[float, float, float]:
    prev_scaled = display_image(prev_image).astype(np.float32)
    current_scaled = display_image(current_image).astype(np.float32)
    shift, _, _ = phase_cross_correlation(prev_scaled, current_scaled, upsample_factor=10)
    return float(shift[0]), float(shift[1]), float(np.hypot(shift[0], shift[1]))


def mask_boundary(mask: np.ndarray) -> np.ndarray:
    if not mask.any():
        return np.zeros_like(mask, dtype=bool)
    dilated = morphology.binary_dilation(mask, morphology.disk(1))
    eroded = morphology.binary_erosion(mask, morphology.disk(1))
    return np.logical_and(dilated, ~eroded)


def position_display_limits(position_label: str, sample_n: int = 12) -> tuple[float, float]:
    subset = metrics_df.loc[
        (metrics_df["position_label"] == position_label)
    ].sort_values("time_index")
    frames = subset["time_index"].astype(int).tolist()
    if not frames:
        return 0.0, 1.0
    if len(frames) <= sample_n:
        sample_frames = frames
    else:
        indices = np.linspace(0, len(frames) - 1, sample_n).round().astype(int)
        sample_frames = [frames[idx] for idx in indices]

    lows = []
    highs = []
    for time_index in sample_frames:
        image = load_phase(position_label, time_index)
        low, high = np.percentile(image, [1.0, 99.0])
        lows.append(low)
        highs.append(high)

    low = float(np.median(lows))
    high = float(np.median(highs))
    if math.isclose(high, low):
        high = low + 1.0
    return low, high


def phase_mask_overlay_rgb(
    phase_image: np.ndarray,
    mask: np.ndarray,
    low: float,
    high: float,
    outline_rgb: tuple[int, int, int] = (0, 255, 255),
) -> np.ndarray:
    scaled = np.clip((phase_image.astype(float) - low) / (high - low), 0, 1)
    rgb = np.repeat((scaled * 255).astype(np.uint8)[..., None], 3, axis=2)
    boundary = mask_boundary(mask)
    if boundary.any():
        rgb[boundary] = np.array(outline_rgb, dtype=np.uint8)
    return rgb


def evenly_spaced_frames(position_label: str, n_frames: int = 4) -> list[int]:
    subset = metrics_df.loc[
        (metrics_df["position_label"] == position_label)
        & (~metrics_df["exclude_from_analysis"])
    ].sort_values("time_index")
    frames = subset["time_index"].astype(int).tolist()
    if not frames:
        return []
    if len(frames) <= n_frames:
        return frames
    indices = np.linspace(0, len(frames) - 1, n_frames).round().astype(int)
    return [frames[idx] for idx in indices]


def select_diverse_candidates(df: pd.DataFrame, n_total: int, max_per_position: int) -> pd.DataFrame:
    selected_rows = []
    counts: dict[str, int] = {}
    for _, row in df.iterrows():
        position_label = row["position_label"]
        if counts.get(position_label, 0) >= max_per_position:
            continue
        counts[position_label] = counts.get(position_label, 0) + 1
        selected_rows.append(row)
        if len(selected_rows) >= n_total:
            break
    return pd.DataFrame(selected_rows)


def persistent_border_touch_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for position_label, subset in df.groupby("position_label"):
        subset = subset.sort_values("time_index").reset_index(drop=True)
        touch_flags = subset["touches_border"].astype(bool).tolist()
        times = subset["time_index"].astype(int).tolist()
        persistent_onset = None
        persistent_count = 0

        for idx, flag in enumerate(touch_flags):
            if flag and all(touch_flags[idx:]):
                persistent_onset = times[idx]
                persistent_count = len(touch_flags) - idx
                break

        if persistent_onset is not None:
            rows.append(
                {
                    "position_label": position_label,
                    "persistent_border_touch_onset": persistent_onset,
                    "persistent_border_touch_count": persistent_count,
                    "border_touch_frame_count": int(sum(touch_flags)),
                }
            )

    return pd.DataFrame(rows).sort_values(
        ["persistent_border_touch_count", "position_label"],
        ascending=[False, True],
    )


## Build QC Watch Lists

This section computes the soft-review score, separates likely mask issues from whole-image shifts, and writes the watch-list tables used below.


In [ ]:
# -------------------------------
# Soft review scoring
# -------------------------------
included_df = metrics_df.loc[~metrics_df["exclude_from_analysis"]].copy()
included_df["second_component_ratio"] = (
    included_df["second_component_area_px"] / included_df["final_mask_area_px"].replace(0, np.nan)
).fillna(0.0)

confidence_score = (
    5.0 * (0.60 - included_df["boundary_prob_mean"]).clip(lower=0)
    + 2.0 * (0.55 - included_df["boundary_prob_p10"]).clip(lower=0)
    + 1.5 * (included_df["area_change_frac_prev"].fillna(0) - 0.18).clip(lower=0)
)
component_penalty = (
    (included_df["second_component_ratio"] - 0.25).clip(lower=0)
) * (
    (included_df["boundary_prob_mean"] < 0.65)
    | (included_df["area_change_frac_prev"].fillna(0) > 0.15)
)
iou_penalty = (
    0.8 * (0.85 - included_df["iou_prev"].fillna(0.85)).clip(lower=0)
) * (
    (included_df["boundary_prob_mean"] < 0.62)
    | (included_df["area_change_frac_prev"].fillna(0) > 0.15)
)
local_boundary_shift_score = (
    0.35 * (included_df["p95_boundary_disp_prev"].fillna(0) - 2.0).clip(lower=0)
    + 0.30 * (included_df["p99_boundary_disp_prev"].fillna(0) - 6.0).clip(lower=0)
    + 0.12 * (included_df["max_boundary_disp_prev"].fillna(0) - 9.0).clip(lower=0)
    + 6.0 * (included_df["missing_prev_frac"].fillna(0) - 0.025).clip(lower=0)
    + 6.0 * (included_df["gained_prev_frac"].fillna(0) - 0.025).clip(lower=0)
)
included_df["soft_review_score"] = (
    confidence_score + component_penalty + iou_penalty + local_boundary_shift_score
)

reason_df = pd.DataFrame(
    {
        "local_boundary_shift": local_boundary_shift_score,
        "low_boundary_confidence": confidence_score,
        "component_split": component_penalty,
        "shape_overlap_drop": iou_penalty,
    }
)
included_df["soft_review_reason"] = reason_df.idxmax(axis=1)
included_df.loc[
    included_df["soft_review_score"] <= 0,
    "soft_review_reason",
] = "stable"

included_df["local_undersegmentation_flag"] = (
    (included_df["p99_boundary_disp_prev"].fillna(0) >= 6.0)
    & (
        (included_df["missing_prev_frac"].fillna(0) >= 0.025)
        | (included_df["gained_prev_frac"].fillna(0) >= 0.025)
        | (included_df["p95_boundary_disp_prev"].fillna(0) >= 2.5)
    )
)

included_df["candidate_label"] = np.select(
    condlist=[
        included_df["touches_border"],
        included_df["soft_review_score"] >= 0.12,
        included_df["used_in_round1_training"],
    ],
    choicelist=["off_frame", "soft_review_flag", "training_frame_reference"],
    default="stable",
)

review_pool = included_df.loc[
    (included_df["candidate_label"] == "soft_review_flag")
    & (~included_df["used_in_round1_training"])
].sort_values(
    ["soft_review_score", "position_label", "time_index"],
    ascending=[False, True, True],
)
general_review_candidates = select_diverse_candidates(review_pool, n_total=18, max_per_position=2)

local_underseg_pool = included_df.loc[
    (~included_df["used_in_round1_training"])
    & (~included_df["touches_border"])
    & (included_df["local_undersegmentation_flag"])
].sort_values(
    ["soft_review_score", "position_label", "time_index"],
    ascending=[False, True, True],
)
local_underseg_examples = select_diverse_candidates(local_underseg_pool, n_total=10, max_per_position=2)

review_candidates_soft_flags = (
    pd.concat([general_review_candidates, local_underseg_examples], ignore_index=True)
    .drop_duplicates(subset=["position_label", "time_index"])
    .sort_values(["soft_review_score", "position_label", "time_index"], ascending=[False, True, True])
    .reset_index(drop=True)
)

shift_rows = []
for _, row in review_candidates_soft_flags.iterrows():
    position_label = row["position_label"]
    time_index = int(row["time_index"])
    prev_time_index = int(row["previous_time_index"]) if pd.notna(row["previous_time_index"]) else None
    if prev_time_index is None:
        shift_rows.append(
            {
                "position_label": position_label,
                "time_index": time_index,
                "global_phase_shift_row_px": np.nan,
                "global_phase_shift_col_px": np.nan,
                "global_phase_shift_mag_px": np.nan,
            }
        )
        continue

    prev_image = load_phase(position_label, prev_time_index)
    current_image = load_phase(position_label, time_index)
    shift_row_px, shift_col_px, shift_mag_px = estimate_global_phase_shift(prev_image, current_image)
    shift_rows.append(
        {
            "position_label": position_label,
            "time_index": time_index,
            "global_phase_shift_row_px": shift_row_px,
            "global_phase_shift_col_px": shift_col_px,
            "global_phase_shift_mag_px": shift_mag_px,
        }
    )

shift_df = pd.DataFrame(shift_rows)
review_candidates_soft_flags = review_candidates_soft_flags.merge(
    shift_df,
    on=["position_label", "time_index"],
    how="left",
)
review_candidates_soft_flags["image_shift_flag"] = (
    review_candidates_soft_flags["global_phase_shift_mag_px"].fillna(0) >= 8.0
)
review_candidates_soft_flags["review_category"] = np.select(
    [
        review_candidates_soft_flags["image_shift_flag"],
        review_candidates_soft_flags["local_undersegmentation_flag"],
    ],
    [
        "image_shift",
        "local_mask_review",
    ],
    default="soft_mask_review",
)

image_shift_flags = review_candidates_soft_flags.loc[
    review_candidates_soft_flags["image_shift_flag"]
].copy()
mask_review_soft_flags = review_candidates_soft_flags.loc[
    ~review_candidates_soft_flags["image_shift_flag"]
].copy()

off_frame_positions = persistent_border_touch_summary(included_df)

review_candidate_path = TABLE_DIR / "03_ilastik_review_candidates_soft_flags.tsv"
local_underseg_path = TABLE_DIR / "03_ilastik_local_undersegmentation_examples.tsv"
image_shift_path = TABLE_DIR / "03_ilastik_image_shift_flags.tsv"
mask_review_path = TABLE_DIR / "03_ilastik_mask_review_soft_flags.tsv"
off_frame_path = TABLE_DIR / "03_ilastik_off_frame_positions.tsv"

review_candidates_soft_flags.to_csv(review_candidate_path, sep="\t", index=False)
local_underseg_examples.to_csv(local_underseg_path, sep="\t", index=False)
image_shift_flags.to_csv(image_shift_path, sep="\t", index=False)
mask_review_soft_flags.to_csv(mask_review_path, sep="\t", index=False)
off_frame_positions.to_csv(off_frame_path, sep="\t", index=False)

display(
    Markdown(
        f'''
        ### Soft Review Flag Set

        After excluding image-junk artifact frames, persistent border-hit windows, and training frames,
        the notebook keeps **{len(review_candidates_soft_flags)}** automatically flagged frames as a
        **soft review set**.

        These frames did **not** fail QC.
        They were flagged because either:

        - the boundary confidence is softer than usual, or
        - the mask boundary shifts locally from one frame to the next in a way that can indicate clipped tips

        The notebook then checks the full phase image for whole-frame translation.
        Large whole-image shifts are split out into a separate `image shift` category so they are not mistaken for cyst-mask failures.
        '''
    )
)

print("Wrote soft review flags:", review_candidate_path)
print("Wrote local undersegmentation examples:", local_underseg_path)
print("Wrote image shift flags:", image_shift_path)
print("Wrote mask-review soft flags:", mask_review_path)
print("Wrote off-frame summary:", off_frame_path)


## How To Read The Worst Remaining Cases

The key evidence is the neighborhood review below.

For each soft review flag, the notebook shows `t-1`, the flagged frame `t`, and `t+1`.
The practical question is simple:

- does the mask stay on the same organoid?
- does it jump abruptly at the flagged frame?
- does the flagged frame look meaningfully worse than its neighbors?
- if a thin protrusion is present, does the mask clip that tip in the flagged frame?

If the flagged frame looks similar to `t-1` and `t+1`, that supports the case that the mask is still accurate enough to use.


### Representative Position Review


In [ ]:
# -------------------------------
# Representative position review figures
# -------------------------------
representative_positions = sorted(
    {
        "Pos1",
        "Pos2",
        "Pos3",
        "Pos7",
        "Pos8",
        "Pos9",
        "Pos31",
        "Pos33",
        "Pos41",
        "Pos56",
        "Pos63",
        *mask_review_soft_flags["position_label"].tolist(),
    }
)

representative_positions


In [ ]:
for position_label in representative_positions:
    frames = evenly_spaced_frames(position_label, n_frames=4)
    if not frames:
        continue

    fig, axes = plt.subplots(len(frames), 3, figsize=(10, 2.8 * len(frames)), constrained_layout=True)
    if len(frames) == 1:
        axes = axes[None, :]

    for row_index, time_index in enumerate(frames):
        metrics_row = metrics_df.loc[
            (metrics_df["position_label"] == position_label)
            & (metrics_df["time_index"] == time_index)
        ].iloc[0]
        phase_image = load_phase(position_label, time_index)
        organoid_probability = load_probability(position_label, time_index)
        mask = load_mask(position_label, time_index)

        axes[row_index, 0].imshow(display_image(phase_image), cmap="gray")
        axes[row_index, 0].set_title(f"{position_label} phase t={format_display_hours_from_index(time_index, 0)}")

        axes[row_index, 1].imshow(organoid_probability, cmap="magma", vmin=0, vmax=1)
        axes[row_index, 1].set_title(
            f"Prob mean={metrics_row['mean_prob_inside_mask']:.2f}\nBoundary={metrics_row['boundary_prob_mean']:.2f}"
        )

        axes[row_index, 2].imshow(display_image(phase_image), cmap="gray")
        draw_mask(axes[row_index, 2], mask)
        axes[row_index, 2].set_title(
            f"Mask area={metrics_row['final_mask_area_frac']:.3f}\nBorder={bool(metrics_row['touches_border'])}"
        )

        for col_index in range(3):
            axes[row_index, col_index].set_xticks([])
            axes[row_index, col_index].set_yticks([])

    out_path = PREVIEW_DIR / f"{position_label}_mask_review.png"
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

print("Wrote representative position review figures to:", PREVIEW_DIR)


### Highest-Scoring Mask Review Flags


In [ ]:
# -------------------------------
# Worst remaining soft-review-flag montage
# -------------------------------
top_candidates = mask_review_soft_flags.head(12).copy()

if not top_candidates.empty:
    fig, axes = plt.subplots(len(top_candidates), 3, figsize=(10, 2.8 * len(top_candidates)), constrained_layout=True)
    if len(top_candidates) == 1:
        axes = axes[None, :]

    for row_index, (_, row) in enumerate(top_candidates.iterrows()):
        position_label = row["position_label"]
        time_index = int(row["time_index"])
        phase_image = load_phase(position_label, time_index)
        organoid_probability = load_probability(position_label, time_index)
        mask = load_mask(position_label, time_index)

        axes[row_index, 0].imshow(display_image(phase_image), cmap="gray")
        axes[row_index, 0].set_title(f"{position_label} t={format_display_hours_from_index(time_index, 0)}")

        axes[row_index, 1].imshow(organoid_probability, cmap="magma", vmin=0, vmax=1)
        axes[row_index, 1].set_title(
            f"Inside={row['mean_prob_inside_mask']:.2f}\nBoundary={row['boundary_prob_mean']:.2f}"
        )

        axes[row_index, 2].imshow(display_image(phase_image), cmap="gray")
        draw_mask(axes[row_index, 2], mask)
        axes[row_index, 2].set_title(
            f"Score={row['soft_review_score']:.2f}\n{row['soft_review_reason']}"
        )

        for col_index in range(3):
            axes[row_index, col_index].set_xticks([])
            axes[row_index, col_index].set_yticks([])

    candidate_out = FIGURE_DIR / "03_ilastik_review_candidates_soft_flags.png"
    fig.savefig(candidate_out, dpi=150)
    display(fig)
    plt.close(fig)
    print("Wrote soft review flag montage:", candidate_out)


### Local Undersegmentation Examples


In [ ]:
# -------------------------------
# Local undersegmentation examples
# -------------------------------
top_local_underseg = local_underseg_examples.head(12).copy()

if not top_local_underseg.empty:
    fig, axes = plt.subplots(len(top_local_underseg), 3, figsize=(10, 2.8 * len(top_local_underseg)), constrained_layout=True)
    if len(top_local_underseg) == 1:
        axes = axes[None, :]

    for row_index, (_, row) in enumerate(top_local_underseg.iterrows()):
        position_label = row["position_label"]
        time_index = int(row["time_index"])
        phase_image = load_phase(position_label, time_index)
        mask = load_mask(position_label, time_index)
        probability = load_probability(position_label, time_index)

        axes[row_index, 0].imshow(display_image(phase_image), cmap="gray")
        axes[row_index, 0].set_title(f"{position_label} t={format_display_hours_from_index(time_index, 0)}")

        axes[row_index, 1].imshow(probability, cmap="magma", vmin=0, vmax=1)
        axes[row_index, 1].set_title(
            f"Boundary p99 shift={row['p99_boundary_disp_prev']:.2f}\nMax shift={row['max_boundary_disp_prev']:.2f}"
        )

        axes[row_index, 2].imshow(display_image(phase_image), cmap="gray")
        draw_mask(axes[row_index, 2], mask)
        axes[row_index, 2].set_title(
            f"Missing={row['missing_prev_frac']:.3f}\nGained={row['gained_prev_frac']:.3f}"
        )

        for col_index in range(3):
            axes[row_index, col_index].set_xticks([])
            axes[row_index, col_index].set_yticks([])

    local_underseg_out = FIGURE_DIR / "03_ilastik_local_undersegmentation_examples.png"
    fig.savefig(local_underseg_out, dpi=150)
    display(fig)
    plt.close(fig)
    print("Wrote local undersegmentation montage:", local_underseg_out)


### Whole-Image Shift Examples


In [ ]:
# -------------------------------
# Whole-image shift examples
# -------------------------------
if not image_shift_flags.empty:
    display(
        Markdown(
            '''
            ### Whole-Image Shift Check

            These frames were first flagged by the mask QC, but then the full phase image showed a large frame-to-frame translation.
            They are better interpreted as image or stage shifts than as cyst-mask failures.
            '''
        )
    )

    fig, axes = plt.subplots(len(image_shift_flags), 3, figsize=(10, 2.8 * len(image_shift_flags)), constrained_layout=True)
    if len(image_shift_flags) == 1:
        axes = axes[None, :]

    for row_index, (_, row) in enumerate(image_shift_flags.iterrows()):
        position_label = row["position_label"]
        time_index = int(row["time_index"])
        prev_time_index = int(row["previous_time_index"])

        prev_phase = load_phase(position_label, prev_time_index)
        current_phase = load_phase(position_label, time_index)
        current_mask = load_mask(position_label, time_index)

        axes[row_index, 0].imshow(display_image(prev_phase), cmap="gray")
        axes[row_index, 0].set_title(f"{position_label} t={format_display_hours_from_index(prev_time_index, 0)}")

        axes[row_index, 1].imshow(display_image(current_phase), cmap="gray")
        axes[row_index, 1].set_title(
            f"t={format_display_hours_from_index(time_index, 0)}\nshift={row['global_phase_shift_mag_px']:.1f}px"
        )

        axes[row_index, 2].imshow(display_image(current_phase), cmap="gray")
        draw_mask(axes[row_index, 2], current_mask)
        axes[row_index, 2].set_title("current mask")

        for col_index in range(3):
            axes[row_index, col_index].set_xticks([])
            axes[row_index, col_index].set_yticks([])

    image_shift_out = FIGURE_DIR / "03_ilastik_image_shift_flags.png"
    fig.savefig(image_shift_out, dpi=150)
    display(fig)
    plt.close(fig)
    print("Wrote image shift montage:", image_shift_out)


### One Representative Flagged Window Per Position


In [ ]:
# -------------------------------
# One representative flagged window per mask-review position
# -------------------------------
representative_flagged_by_position = (
    mask_review_soft_flags
    .sort_values(["position_label", "soft_review_score", "time_index"], ascending=[True, False, True])
    .groupby("position_label", as_index=False)
    .first()
    .sort_values(["soft_review_score", "position_label"], ascending=[False, True])
    .reset_index(drop=True)
)

if not representative_flagged_by_position.empty:
    display(
        Markdown(
            '''
            ### Representative Flagged Window Per Position

            This section shows one representative flagged window for every mask-review position.
            It is meant to answer a simple question: when a position gets flagged at all, what is the single best example of why?
            '''
        )
    )

    fig, axes = plt.subplots(
        len(representative_flagged_by_position),
        3,
        figsize=(10, 2.8 * len(representative_flagged_by_position)),
        constrained_layout=True,
    )
    if len(representative_flagged_by_position) == 1:
        axes = axes[None, :]

    for row_index, (_, row) in enumerate(representative_flagged_by_position.iterrows()):
        position_label = row["position_label"]
        time_index = int(row["time_index"])
        position_subset = metrics_df.loc[
            metrics_df["position_label"] == position_label
        ].sort_values("time_index")
        available_times = set(position_subset["time_index"].astype(int).tolist())
        neighborhood_times = [time_index - 1, time_index, time_index + 1]
        neighborhood_times = [t for t in neighborhood_times if t in available_times]

        for col_index, neighbor_time in enumerate(neighborhood_times):
            phase_image = load_phase(position_label, neighbor_time)
            mask = load_mask(position_label, neighbor_time)
            metric_row = metrics_df.loc[
                (metrics_df["position_label"] == position_label)
                & (metrics_df["time_index"] == neighbor_time)
            ].iloc[0]
            axes[row_index, col_index].imshow(display_image(phase_image), cmap="gray")
            draw_mask(axes[row_index, col_index], mask)
            title = f"{position_label} t={format_display_hours_from_index(neighbor_time, 0)}"
            if neighbor_time == time_index:
                title += (
                    f"\n{row['soft_review_reason']}"
                    f" score={row['soft_review_score']:.2f}"
                )
            else:
                title += (
                    f"\np99 shift={metric_row['p99_boundary_disp_prev']:.2f}"
                    if pd.notna(metric_row["p99_boundary_disp_prev"])
                    else f"\nboundary={metric_row['boundary_prob_mean']:.2f}"
                )
            axes[row_index, col_index].set_title(title)
            axes[row_index, col_index].set_xticks([])
            axes[row_index, col_index].set_yticks([])

        for col_index in range(len(neighborhood_times), 3):
            axes[row_index, col_index].axis("off")

    representative_out = FIGURE_DIR / "03_ilastik_mask_review_by_position.png"
    fig.savefig(representative_out, dpi=150)
    display(fig)
    plt.close(fig)
    print("Wrote one-per-position mask-review montage:", representative_out)


### Pos0 Clipped-Tip Case Study


In [ ]:
# -------------------------------
# Pos0 clipped-tip case study
# -------------------------------
pos0_cases = included_df.loc[
    (included_df["position_label"] == "Pos0")
    & (~included_df["used_in_round1_training"])
    & (included_df["local_undersegmentation_flag"])
].sort_values("time_index")

pos0_focus = pos0_cases.loc[pos0_cases["time_index"] >= 150].copy()
if pos0_focus.empty:
    pos0_focus = pos0_cases.copy()

if not pos0_focus.empty:
    display(
        Markdown(
            '''
            ### Pos0 Clipped-Tip Check

            `Pos0` was a human-spotted example where the lower-right curled tip gets clipped in the middle of the movie.
            The local boundary-shift QC now surfaces that same kind of frame directly.
            '''
        )
    )

    focus_rows = (
        pos0_focus.sort_values(["soft_review_score", "time_index"], ascending=[False, True])
        .head(4)
        .sort_values("time_index")
        .copy()
    )
    fig, axes = plt.subplots(len(focus_rows), 3, figsize=(10, 2.8 * len(focus_rows)), constrained_layout=True)
    if len(focus_rows) == 1:
        axes = axes[None, :]

    for row_index, (_, row) in enumerate(focus_rows.iterrows()):
        time_index = int(row["time_index"])
        neighborhood_times = [t for t in [time_index - 1, time_index, time_index + 1] if 0 <= t <= 273]

        for col_index, neighbor_time in enumerate(neighborhood_times):
            phase_image = load_phase("Pos0", neighbor_time)
            mask = load_mask("Pos0", neighbor_time)
            metric_row = metrics_df.loc[
                (metrics_df["position_label"] == "Pos0")
                & (metrics_df["time_index"] == neighbor_time)
            ].iloc[0]
            axes[row_index, col_index].imshow(display_image(phase_image), cmap="gray")
            draw_mask(axes[row_index, col_index], mask)
            title = f"Pos0 t={format_display_hours_from_index(neighbor_time, 0)}"
            if neighbor_time == time_index:
                title += (
                    f"\np99 shift={row['p99_boundary_disp_prev']:.2f}"
                    f" miss={row['missing_prev_frac']:.3f}"
                    f" gain={row['gained_prev_frac']:.3f}"
                )
            else:
                title += f"\np99 shift={metric_row['p99_boundary_disp_prev']:.2f}"
            axes[row_index, col_index].set_title(title)
            axes[row_index, col_index].set_xticks([])
            axes[row_index, col_index].set_yticks([])

        for col_index in range(len(neighborhood_times), 3):
            axes[row_index, col_index].axis("off")

    pos0_out = FIGURE_DIR / "03_ilastik_pos0_clipped_tip_check.png"
    fig.savefig(pos0_out, dpi=150)
    display(fig)
    plt.close(fig)
    print("Wrote Pos0 clipped-tip check:", pos0_out)


### Flagged Neighborhood Review

These views are the most direct check of whether the flagged frame actually looks worse than its immediate neighbors.


In [ ]:
# -------------------------------
# Soft review flag neighborhood review
# -------------------------------
candidate_neighborhood_dir = PREVIEW_DIR / "review_candidate_neighborhoods"
candidate_neighborhood_dir.mkdir(parents=True, exist_ok=True)

neighborhood_rows = []
for _, row in mask_review_soft_flags.iterrows():
    position_label = row["position_label"]
    time_index = int(row["time_index"])
    position_subset = metrics_df.loc[
        (metrics_df["position_label"] == position_label)
    ].sort_values("time_index")
    available_times = set(position_subset["time_index"].astype(int).tolist())
    neighborhood_times = [time_index - 1, time_index, time_index + 1]
    neighborhood_times = [t for t in neighborhood_times if t in available_times]
    neighborhood_rows.append(
        {
            "position_label": position_label,
            "time_index": time_index,
            "neighborhood_times": neighborhood_times,
        }
    )

    fig, axes = plt.subplots(1, len(neighborhood_times), figsize=(3.4 * len(neighborhood_times), 3.4), constrained_layout=True)
    if len(neighborhood_times) == 1:
        axes = [axes]

    for ax, neighbor_time in zip(axes, neighborhood_times):
        phase_image = load_phase(position_label, neighbor_time)
        mask = load_mask(position_label, neighbor_time)
        metric_row = metrics_df.loc[
            (metrics_df["position_label"] == position_label)
            & (metrics_df["time_index"] == neighbor_time)
        ].iloc[0]
        ax.imshow(display_image(phase_image), cmap="gray")
        draw_mask(ax, mask)
        title = f"{position_label} t={format_display_hours_from_index(neighbor_time, 0)}"
        if neighbor_time == time_index:
            title += f"\nflagged {row['soft_review_reason']}"
        else:
            title += (
                f"\nshift p99={metric_row['p99_boundary_disp_prev']:.2f}"
                if pd.notna(metric_row["p99_boundary_disp_prev"])
                else f"\nBoundary={metric_row['boundary_prob_mean']:.2f}"
            )
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])

    out_path = candidate_neighborhood_dir / f"{position_label}_t{time_index:03d}_neighborhood.png"
    fig.savefig(out_path, dpi=150)
    display(fig)
    plt.close(fig)

neighborhood_df = pd.DataFrame(neighborhood_rows)
neighborhood_table_path = TABLE_DIR / "03_ilastik_review_candidate_neighborhoods.tsv"
neighborhood_df.to_csv(neighborhood_table_path, sep="\t", index=False)
print("Wrote soft review neighborhood previews to:", candidate_neighborhood_dir)
print("Wrote soft review neighborhood table:", neighborhood_table_path)


In [ ]:
# -------------------------------
# Combined neighborhood montage
# -------------------------------
if not mask_review_soft_flags.empty:
    fig, axes = plt.subplots(len(mask_review_soft_flags), 3, figsize=(10, 2.8 * len(mask_review_soft_flags)), constrained_layout=True)
    if len(mask_review_soft_flags) == 1:
        axes = axes[None, :]

    for row_index, (_, row) in enumerate(mask_review_soft_flags.iterrows()):
        position_label = row["position_label"]
        time_index = int(row["time_index"])
        position_subset = metrics_df.loc[
            (metrics_df["position_label"] == position_label)
        ].sort_values("time_index")
        available_times = set(position_subset["time_index"].astype(int).tolist())
        neighborhood_times = [time_index - 1, time_index, time_index + 1]
        neighborhood_times = [t for t in neighborhood_times if t in available_times]

        for col_index, neighbor_time in enumerate(neighborhood_times):
            phase_image = load_phase(position_label, neighbor_time)
            mask = load_mask(position_label, neighbor_time)
            metric_row = metrics_df.loc[
                (metrics_df["position_label"] == position_label)
                & (metrics_df["time_index"] == neighbor_time)
            ].iloc[0]
            axes[row_index, col_index].imshow(display_image(phase_image), cmap="gray")
            draw_mask(axes[row_index, col_index], mask)
            title = f"{position_label} t={format_display_hours_from_index(neighbor_time, 0)}"
            if neighbor_time == time_index:
                title += f"\n{row['soft_review_reason']}"
            else:
                title += (
                    f"\np99 shift {metric_row['p99_boundary_disp_prev']:.2f}"
                    if pd.notna(metric_row["p99_boundary_disp_prev"])
                    else f"\nboundary {metric_row['boundary_prob_mean']:.2f}"
                )
            axes[row_index, col_index].set_title(title)
            axes[row_index, col_index].set_xticks([])
            axes[row_index, col_index].set_yticks([])

        for col_index in range(len(neighborhood_times), 3):
            axes[row_index, col_index].axis("off")

    neighborhood_out = FIGURE_DIR / "03_ilastik_review_candidate_neighborhoods.png"
    fig.savefig(neighborhood_out, dpi=150)
    display(fig)
    plt.close(fig)
    print("Wrote soft review neighborhood montage:", neighborhood_out)


## Summary


In [ ]:
# -------------------------------
# Plain-language interpretation
# -------------------------------
review_summary = mask_review_soft_flags.copy()
mean_score = review_summary["soft_review_score"].mean() if not review_summary.empty else float("nan")
local_underseg_count = int(local_underseg_examples.shape[0])
image_shift_count = int(image_shift_flags.shape[0])

interpretation_lines = [
    "### Image-First Interpretation",
    "",
    "This notebook is not trying to prove the masks are perfect.",
    "It is asking a narrower question: after excluding obvious image junk and persistent border-hit windows, do the worst remaining cases still look usable?",
    "",
    f"- mask-review soft flags shown here: {len(review_summary)}",
    f"- image-shift flags split out separately: {image_shift_count}",
    f"- local undersegmentation watch-list frames: {local_underseg_count}",
    "- these are the worst remaining non-artifact, non-border-hit, non-training frames by the automated QC score",
    f"- average soft-review score: {mean_score:.2f}" if not math.isnan(mean_score) else "- no soft review flags found",
    "",
    "What to look for in the neighborhoods above:",
    "- whether the mask stays on the same organoid across t-1, t, and t+1",
    "- whether the flagged frame jumps abruptly compared with its neighbors",
    "- whether the mask edge clips a thin protrusion or curls inward on one side",
    "- whether the mask edge looks grossly misplaced, not just slightly conservative",
    "",
    "The key new QC check here is local boundary shift.",
    "That makes the notebook sensitive to small undersegmentation errors, like a tip being clipped off while the main body of the cyst remains tracked.",
    "My read of these examples is still that the flagged frames are mostly borderline or conservative, not catastrophic failures.",
    "That supports proceeding without a mandatory second manual labeling round, while still allowing a very small refinement round if you want it.",
]
display(Markdown("\n".join(interpretation_lines)))


## Save Stage Outputs

The next export writes one RGB multipage TIFF per position so the entire cyst-tracking movie can be reviewed in Fiji.

Each stack contains:

- phase / brightfield image
- cyst mask boundary overlaid in cyan

These stacks are intended for manual inspection in Fiji across the full timelapse for each position.


### Write Fiji Overlay Stacks


In [ ]:
# -------------------------------
# Export phase + cyst-boundary stacks for Fiji
# -------------------------------
stack_rows = []
position_labels = sorted(metrics_df["position_label"].unique())

for position_label in position_labels:
    subset = metrics_df.loc[
        metrics_df["position_label"] == position_label
    ].sort_values("time_index")
    time_indices = subset["time_index"].astype(int).tolist()
    low, high = position_display_limits(position_label)

    stack_frames = []
    for time_index in time_indices:
        phase_image = load_phase(position_label, time_index)
        mask = load_mask(position_label, time_index)
        overlay = phase_mask_overlay_rgb(phase_image, mask, low=low, high=high)
        stack_frames.append(overlay)

    stack = np.stack(stack_frames, axis=0)
    stack_path = STACK_DIR / f"{position_label}_phase_mask_overlay_stack.tif"
    tiff.imwrite(
        stack_path,
        stack,
        imagej=True,
        compression="zlib",
        metadata={"axes": "TYXS"},
    )

    stack_rows.append(
        {
            "position_label": position_label,
            "frame_count": len(time_indices),
            "stack_path": str(stack_path),
            "display_low": low,
            "display_high": high,
        }
    )

stack_manifest = pd.DataFrame(stack_rows)
stack_manifest_path = TABLE_DIR / "03_ilastik_phase_mask_overlay_stacks.tsv"
stack_manifest.to_csv(stack_manifest_path, sep="\t", index=False)

display(display_time_df(stack_manifest.head()))
print("Wrote overlay stacks to:", STACK_DIR)
print("Wrote stack manifest to:", stack_manifest_path)


## Review Off-Frame Growth

This figure is kept separate from the mask-accuracy review because these are analysis-scope issues rather than ilastik training failures.


In [ ]:
# -------------------------------
# Off-frame review montage
# -------------------------------
off_frame_examples = ["Pos63", "Pos41", "Pos38"]
off_frame_examples = [pos for pos in off_frame_examples if pos in set(off_frame_positions["position_label"])]

if off_frame_examples:
    fig, axes = plt.subplots(len(off_frame_examples), 4, figsize=(12, 3.0 * len(off_frame_examples)), constrained_layout=True)
    if len(off_frame_examples) == 1:
        axes = axes[None, :]

    for row_index, position_label in enumerate(off_frame_examples):
        position_subset = metrics_df.loc[
            (metrics_df["position_label"] == position_label)
            & (~metrics_df["exclude_from_analysis"])
        ].sort_values("time_index")

        border_frames = position_subset.loc[position_subset["touches_border"], "time_index"].astype(int).tolist()
        persistent_onset = int(
            off_frame_positions.loc[
                off_frame_positions["position_label"] == position_label,
                "persistent_border_touch_onset",
            ].iloc[0]
        )
        candidate_frames = sorted(
            {
                int(position_subset["time_index"].iloc[0]),
                persistent_onset,
                border_frames[len(border_frames) // 2],
                border_frames[-1],
            }
        )

        for col_index, time_index in enumerate(candidate_frames[:4]):
            phase_image = load_phase(position_label, time_index)
            mask = load_mask(position_label, time_index)
            axes[row_index, col_index].imshow(display_image(phase_image), cmap="gray")
            draw_mask(axes[row_index, col_index], mask)
            axes[row_index, col_index].set_title(f"{position_label} t={format_display_hours_from_index(time_index, 0)}")
            axes[row_index, col_index].set_xticks([])
            axes[row_index, col_index].set_yticks([])

    off_frame_out = FIGURE_DIR / "03_ilastik_off_frame_review.png"
    fig.savefig(off_frame_out, dpi=150)
    display(fig)
    plt.close(fig)
    print("Wrote off-frame review montage:", off_frame_out)


## Reading The Outputs

The main files to inspect after execution are:

- per-position review PNGs in `results/previews/03_ilastik_mask_postprocessing_qc/`
- `results/figures/03/03_ilastik_review_candidates_soft_flags.png`
- `results/figures/03/03_ilastik_image_shift_flags.png`
- `results/figures/03/03_ilastik_mask_review_by_position.png`
- `results/figures/03/03_ilastik_local_undersegmentation_examples.png`
- `results/figures/03/03_ilastik_pos0_clipped_tip_check.png`
- `results/figures/03/03_ilastik_review_candidate_neighborhoods.png`
- `results/figures/03/03_ilastik_off_frame_review.png`
- `results/tables/03_ilastik_review_candidates_soft_flags.tsv`
- `results/tables/03_ilastik_image_shift_flags.tsv`
- `results/tables/03_ilastik_mask_review_soft_flags.tsv`
- `results/tables/03_ilastik_local_undersegmentation_examples.tsv`
- `results/tables/03_ilastik_phase_mask_overlay_stacks.tsv`
- `results/tables/03_ilastik_off_frame_positions.tsv`
- `results/ilastik/qc/phase_mask_overlay_stacks/full_dataset_v1/`

The most important biological decision after this notebook is whether late `border-touching` windows should be excluded from aggregate quantification once the organoid grows out of frame.
